# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 291.81it/s]


2026-09-07 07:34:09.501 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:903 - Data batch-empirical estimation of propensity score.


2026-09-07 07:34:09.509 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:952 - Data prediction of expected reward based on gbm model.


2026-09-07 07:34:10.934 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1069 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-09-07 07:34:46.712 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-09-07 07:34:47.106 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-09-07 07:34:47.496 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.520643,0.487263,0.555084,0.017147,b-ipw,reward_0
1,0.482209,0.481820,0.482606,0.000199,dm,reward_0
2,0.510520,0.479585,0.542756,0.016197,dr,reward_0
3,0.482209,0.481818,0.482602,0.000198,dros-opt,reward_0
4,0.510520,0.477823,0.541809,0.016185,dros-pess,reward_0
5,0.510186,0.477554,0.542630,0.016629,ipw,reward_0
6,0.509763,0.476393,0.543979,0.017202,rep,reward_0
7,0.510540,0.478556,0.540905,0.016042,sndr,reward_0
8,0.510551,0.477570,0.543122,0.016785,snips,reward_0
9,0.510520,0.478648,0.541610,0.015922,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 280.16it/s]


2026-09-07 07:34:48.055 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1301 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<09:53,  1.68it/s]

...

2026-09-07 07:35:44.911 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-09-07 07:35:45.315 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-09-07 07:35:45.721 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.517638,0.485809,0.550509,0.016412,b-ipw,reward_0
1,0.482285,0.481923,0.482656,0.000187,dm,reward_0
2,0.513505,0.482171,0.544736,0.016013,dr,reward_0
3,0.482285,0.481924,0.482657,0.000187,dros-opt,reward_0
4,0.513505,0.482428,0.545096,0.015893,dros-pess,reward_0
5,0.512217,0.481163,0.545172,0.016180,ipw,reward_0
6,0.512837,0.481309,0.545618,0.016438,rep,reward_0
7,0.513590,0.482630,0.544504,0.015814,sndr,reward_0
8,0.513613,0.482901,0.545699,0.016025,snips,reward_0
9,0.513505,0.482593,0.544618,0.015918,sg-dr,reward_0
